In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset"

In [3]:
import os

print(os.listdir("/content/dataset"))

['styles.csv', 'images', 'myntradataset']


In [4]:
import pandas as pd

df = pd.read_csv("/content/dataset/styles.csv", on_bad_lines="skip")

print(df.shape)
df.head()

(44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
men_tshirt_df = df[
    (df["gender"] == "Men") &
    (df["articleType"] == "Tshirts")
].copy()

print("Total Men's T-shirts:", len(men_tshirt_df))

Total Men's T-shirts: 5243


In [6]:
import os

IMAGE_DIR = "/content/dataset/images"

men_tshirt_df["image_path"] = men_tshirt_df["id"].apply(
    lambda x: os.path.join(IMAGE_DIR, str(x) + ".jpg")
)

men_tshirt_df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,/content/dataset/images/53759.jpg
5,1855,Men,Apparel,Topwear,Tshirts,Grey,Summer,2011.0,Casual,Inkfruit Mens Chain Reaction T-shirt,/content/dataset/images/1855.jpg
27,7990,Men,Apparel,Topwear,Tshirts,Navy Blue,Fall,2011.0,Sports,Fila Men's Round Neck Navy Blue T-shirt,/content/dataset/images/7990.jpg
55,5891,Men,Apparel,Topwear,Tshirts,Black,Summer,2011.0,Casual,Puma Men's Stripe Polo Black T-shirt,/content/dataset/images/5891.jpg
59,10866,Men,Apparel,Topwear,Tshirts,Red,Fall,2011.0,Casual,Wrangler Men Motor Rider Red T-Shirts,/content/dataset/images/10866.jpg


In [7]:
men_tshirt_df["exists"] = men_tshirt_df["image_path"].apply(os.path.exists)

print(men_tshirt_df["exists"].value_counts())

exists
True     5242
False       1
Name: count, dtype: int64


In [8]:
men_tshirt_df = men_tshirt_df[men_tshirt_df["exists"] == True].copy()
men_tshirt_df = men_tshirt_df.drop(columns=["exists"])

print("Final Men's T-shirt images:", len(men_tshirt_df))

Final Men's T-shirt images: 5242


In [9]:
men_tshirt_df.to_csv("/content/drive/MyDrive/mens_tshirt_dataframe.csv", index=False)

print("Clean DataFrame saved successfully")

Clean DataFrame saved successfully


In [10]:
from sklearn.model_selection import train_test_split

historical_df, new_market_df = train_test_split(
    men_tshirt_df,
    train_size=3145,
    random_state=42
)

print("Historical:", len(historical_df))
print("New Market:", len(new_market_df))

Historical: 3145
New Market: 2097


In [11]:

required_preferred = 1468
required_not_preferred = 1677

preferred_colours = ["Black", "Grey", "Navy Blue", "White", "Blue"]

preferred_pool = historical_df[
    historical_df["baseColour"].isin(preferred_colours)
].copy()

not_preferred_pool = historical_df[
    ~historical_df["baseColour"].isin(preferred_colours)
].copy()

preferred_final = preferred_pool.sample(n=required_preferred, random_state=42)

remaining_needed = required_not_preferred - len(not_preferred_pool)

extra_not_preferred = preferred_pool.drop(preferred_final.index).sample(
    n=remaining_needed,
    random_state=42
)

not_preferred_final = pd.concat(
    [not_preferred_pool, extra_not_preferred],
    ignore_index=True
)

preferred_final["label"] = 1
not_preferred_final["label"] = 0

preference_df = pd.concat(
    [preferred_final, not_preferred_final],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(preference_df["label"].value_counts())
print("Final Historical Dataset:", len(preference_df))

label
0    1677
1    1468
Name: count, dtype: int64
Final Historical Dataset: 3145


In [17]:
preferred_df = preference_df[preference_df["label"] == 1].copy()
not_preferred_df = preference_df[preference_df["label"] == 0].copy()

train_preferred = preferred_df.sample(n=1049, random_state=42)
test_preferred = preferred_df.drop(train_preferred.index)

train_not_preferred = not_preferred_df.sample(n=1198, random_state=42)
test_not_preferred = not_preferred_df.drop(train_not_preferred.index)

train_df = pd.concat(
    [train_preferred, train_not_preferred],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

test_df = pd.concat(
    [test_preferred, test_not_preferred],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Train Preferred:", len(train_preferred))
print("Train Not Preferred:", len(train_not_preferred))
print("Test Preferred:", len(test_preferred))
print("Test Not Preferred:", len(test_not_preferred))

print("\nFinal Train Dataset:", len(train_df))
print("Final Test Dataset:", len(test_df))

print("\nTrain Labels:")
print(train_df["label"].value_counts())

print("\nTest Labels:")
print(test_df["label"].value_counts())

Train Preferred: 1049
Train Not Preferred: 1198
Test Preferred: 419
Test Not Preferred: 479

Final Train Dataset: 2247
Final Test Dataset: 898

Train Labels:
label
0    1198
1    1049
Name: count, dtype: int64

Test Labels:
label
0    479
1    419
Name: count, dtype: int64


In [20]:
!pip install -q transformers torch pillow tqdm scikit-learn

import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model.eval()

print("CLIP Model Loaded Successfully!")

Device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP Model Loaded Successfully!


In [22]:
def make_embeddings_batch(dataframe, batch_size=32):

    embeddings = []

    images = dataframe["image_path"].tolist()

    for i in tqdm(range(0, len(images), batch_size)):

        batch_paths = images[i:i+batch_size]

        batch_images = [
            Image.open(p).convert("RGB") for p in batch_paths
        ]

        inputs = clip_processor(
            images=batch_images,
            return_tensors="pt"
        )

        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():

            outputs = clip_model.vision_model(pixel_values=pixel_values)

            pooled = outputs.pooler_output

            emb = clip_model.visual_projection(pooled)

            emb = emb / emb.norm(dim=-1, keepdim=True)

        embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings)

In [23]:
train_embeddings = make_embeddings_batch(train_df, batch_size=32)
test_embeddings = make_embeddings_batch(test_df, batch_size=32)
new_market_embeddings = make_embeddings_batch(new_market_df, batch_size=32)

print(train_embeddings.shape)
print(test_embeddings.shape)
print(new_market_embeddings.shape)

100%|██████████| 66/66 [00:12<00:00,  5.25it/s]

(2247, 512)
(898, 512)
(2097, 512)


In [24]:
from sklearn.linear_model import LogisticRegression

preference_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

preference_model.fit(
    train_embeddings,
    train_df["label"]
)

print("Preference model trained successfully!")

Preference model trained successfully!


In [35]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_predictions = preference_model.predict(test_embeddings)

print("Accuracy:", accuracy_score(test_df["label"], test_predictions))

print("\nClassification Report:")
print(classification_report(test_df["label"], test_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(test_df["label"], test_predictions))

Accuracy: 0.734966592427617

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.72      0.74       479
           1       0.70      0.75      0.73       419

    accuracy                           0.73       898
   macro avg       0.74      0.74      0.73       898
weighted avg       0.74      0.73      0.74       898


Confusion Matrix:
[[344 135]
 [103 316]]


In [26]:
test_scores = preference_model.predict_proba(
    test_embeddings
)[:, 1]

test_df["preference_score"] = test_scores

print(test_df[[
    "id",
    "productDisplayName",
    "baseColour",
    "preference_score"
]].head())

      id                     productDisplayName baseColour  preference_score
0  25028       Nike Men Speed Fly Black T-shirt      Black          0.690791
1  30646     Nike Men Barca Logo Yellow T-shirt     Yellow          0.249592
2   8318             Wildcraft Men Grey T-shirt       Grey          0.500555
3   7525      Nike Men's Round Neck Red T-Shirt        Red          0.417229
4  30627  Nike Men Kobe Old Master Grey T-shirt       Grey          0.639070


In [34]:
new_scores = preference_model.predict_proba(
    new_market_embeddings
)[:,1]
new_market_df["prefeence_score"] = new_scores

print(test_df[[
    "id",
    "productDisplayName",
    "baseColour",
    "preference_score"
]].head())


      id                     productDisplayName baseColour  preference_score
0  25028       Nike Men Speed Fly Black T-shirt      Black          0.690791
1  30646     Nike Men Barca Logo Yellow T-shirt     Yellow          0.249592
2   8318             Wildcraft Men Grey T-shirt       Grey          0.500555
3   7525      Nike Men's Round Neck Red T-Shirt        Red          0.417229
4  30627  Nike Men Kobe Old Master Grey T-shirt       Grey          0.639070
